In [1]:
import os
import sys
import datetime

sys.path.insert(0, "/mnt/storage_C4/gaussian_football")
sys.path.insert(0, "/mnt/storage_C4/gaussian_football/preprocessing")
sys.path.insert(0, "/mnt/storage_C4/gaussian_football/models/nn")

import random
import numpy as np
import torch
import torch.nn as nn
from tqdm import tqdm
import torchvision
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.tensorboard import SummaryWriter
from cnnbilstm import CNNBiLSTM
from resnetlstm import ResNetLSTM
from loaders import build_mel_dataloader, build_video_dataloader

In [2]:
print("Versão do Torch:", torch.__version__)
print("Versão do Torchvision:", torchvision.__version__)
print(f"CUDA disponível: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"Versão do CUDA: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")

seed = 435
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Semente: {seed}")
print(f"Dispositivo: {device}")

Versão do Torch: 2.5.1+cu121
Versão do Torchvision: 0.20.1+cu121
CUDA disponível: True
Versão do CUDA: 12.1
GPU: NVIDIA RTX A4500
Semente: 435
Dispositivo: cuda


In [ ]:
CSV_PATH = "/mnt/storage_C4/gaussian_football/data/labels/labels_all.csv"

train_mel_loader = build_mel_dataloader(
    csv_path=CSV_PATH,
    split="train",
    batch_size=4,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
)

valid_mel_loader = build_mel_dataloader(
    csv_path=CSV_PATH,
    split="valid",
    batch_size=4,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)

train_video_loader = build_video_dataloader(
    csv_path=CSV_PATH,
    split="train",
    batch_size=4,
    shuffle=True,
    num_workers=4,
    is_grayscale=True,
    pin_memory=True,
)

valid_video_loader = build_video_dataloader(
    csv_path=CSV_PATH,
    split="valid",
    batch_size=4,
    shuffle=False,
    num_workers=4,
    is_grayscale=True,
    pin_memory=True,
)

Dataset: 36304/37620 exemplos válidos.
Dataset: 11224/12540 exemplos válidos.


In [ ]:
def calculate_ccc(y_true, y_pred):
    """Calcula o Concordance Correlation Coefficient (CCC)."""
    mean_true = np.mean(y_true)
    mean_pred = np.mean(y_pred)
    var_true = np.var(y_true)
    var_pred = np.var(y_pred)

    covariance = np.mean((y_true - mean_true) * (y_pred - mean_pred))
    numerator = 2 * covariance
    denominator = var_true + var_pred + (mean_true - mean_pred) ** 2

    return 0.0 if denominator == 0 else numerator / denominator

In [ ]:
def train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    scheduler,
    epochs=10,
    device="cuda",
    patience=5,
    checkpoint_path="best_model.pth",
):
    """
    Treina a rede e a avalia a cada época usando o CCC como métrica principal.

    Argumentos:
        model:            Modelo PyTorch.
        train_loader:     DataLoader para os dados de treino.
        val_loader:       DataLoader para os dados de validação.
        optimizer:        Otimizador (ex: AdamW).
        criterion:        Função de perda (ex: nn.MSELoss()).
        scheduler:        Scheduler de learning rate (ex: ReduceLROnPlateau).
        epochs:           Número máximo de épocas.
        device:           Dispositivo de treino ("cuda" ou "cpu").
        patience:         Épocas sem melhora no CCC antes de acionar early stopping.
        checkpoint_path:  Caminho para salvar o melhor modelo.
    """

    model.to(device)
    best_val_ccc = -1.0 # CCC varia de - 1 a 1, quanto mais perto de 1, melhor
    epochs_no_improve = 0

    current_time = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    log_dir = os.path.join("runs", f"arousal_{current_time}")
    writer = SummaryWriter(log_dir=log_dir)
    print(f"TensorBoard: {log_dir}")

    for epoch in range(epochs):

        # ------------------------------------------------------------------ #
        # TREINO
        # ------------------------------------------------------------------ #
        model.train()
        train_loss = 0.0

        for videos, masks, targets in tqdm(
            train_loader, desc=f"Época {epoch+1}/{epochs} [treino]"
        ):
            videos = videos.to(device)
            targets = targets.to(device)

            optimizer.zero_grad()
            outputs = model(videos).squeeze(1)
            loss = criterion(outputs, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item() * videos.size(0)

        train_loss /= len(train_loader.dataset)

        # ------------------------------------------------------------------ #
        # VALIDAÇÃO
        # ------------------------------------------------------------------ #
        model.eval()
        val_loss = 0.0
        all_true = []
        all_pred = []

        with torch.no_grad():
            for videos, masks, targets in tqdm(
                val_loader, desc=f"Época {epoch+1}/{epochs} [val]"
            ):
                videos = videos.to(device)
                targets = targets.to(device)

                outputs = model(videos).squeeze(1)
                loss = criterion(outputs, targets)
                val_loss += loss.item() * videos.size(0)

                all_true.extend(targets.cpu().numpy())
                all_pred.extend(outputs.cpu().numpy())

        val_loss /= len(val_loader.dataset)
        val_mae = np.mean(np.abs(np.array(all_true) - np.array(all_pred)))
        val_ccc = calculate_ccc(np.array(all_true), np.array(all_pred))

        scheduler.step(val_loss)

        # ------------------------------------------------------------------ #
        # TENSORBOARD
        # ------------------------------------------------------------------ #
        writer.add_scalar("Loss/train", train_loss, epoch)
        writer.add_scalar("Loss/val", val_loss, epoch)
        writer.add_scalar("Metrics/MAE", val_mae, epoch)
        writer.add_scalar("Metrics/CCC", val_ccc, epoch)
        writer.add_scalar("LR", optimizer.param_groups[0]["lr"], epoch)

        print(
            f"Época [{epoch+1}/{epochs}]"
            f" | Train Loss: {train_loss:.4f}"
            f" | Val Loss: {val_loss:.4f}"
            f" | Val MAE: {val_mae:.4f}"
            f" | Val CCC: {val_ccc:.4f}"
            f" | LR: {optimizer.param_groups[0]['lr']:.2e}"
        )

        # ------------------------------------------------------------------ #
        # CHECKPOINT + EARLY STOPPING
        # ------------------------------------------------------------------ #
        if val_ccc > best_val_ccc:
            best_val_ccc = val_ccc
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "best_val_ccc": best_val_ccc,
            }, checkpoint_path)
            print(f"Novo melhor modelo salvo! (CCC: {best_val_ccc:.4f})")
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            print(f"Sem melhora por {epochs_no_improve}/{patience} épocas.")
            if epochs_no_improve >= patience:
                print("Early stopping ativado.")
                break

    writer.close()
    print(f"\nTreinamento concluído. Melhor CCC: {best_val_ccc:.4f}")

In [8]:
model = CNNBiLSTM(
    in_channels=1,
    frame_step=15,
    use_dropout=True,
    use_batch_norm=True,
    dropout_p=0.3,
).to(device)

criterion = nn.MSELoss()
optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.5)

os.makedirs("/mnt/storage_C4/gaussian_football/models/checkpoints", exist_ok=True)

print(f"Parâmetros treináveis: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Parâmetros treináveis: 11,472,705


In [9]:
train_model(
    model=model,
    train_loader=train_video_loader,
    val_loader=valid_video_loader,
    optimizer=optimizer,
    criterion=criterion,
    scheduler=scheduler,
    epochs=10,
    device=device,
    patience=5,
    checkpoint_path="/mnt/storage_C4/gaussian_football/models/checkpoints/best_model.pth",
)

TensorBoard: runs/arousal_20260605-182452


Época 1/10 [val]: 100%|██████████| 3135/3135 [24:59<00:00,  2.09it/s]


Época [1/10] | Train Loss: 0.0258 | Val Loss: 0.0211 | Val MAE: 0.0339 | Val CCC: 0.0101 | LR: 1.00e-04
Novo melhor modelo salvo! (CCC: 0.0101)


Época 2/10 [val]: 100%|██████████| 3135/3135 [24:43<00:00,  2.11it/s]


Época [2/10] | Train Loss: 0.0161 | Val Loss: 0.0210 | Val MAE: 0.0304 | Val CCC: 0.0024 | LR: 1.00e-04
Sem melhora por 1/5 épocas.


...

Época [4/10] | Train Loss: 0.0160 | Val Loss: 0.0208 | Val MAE: 0.0380 | Val CCC: -0.0006 | LR: 1.00e-04
Sem melhora por 3/5 épocas.


Época 5/10 [val]: 100%|██████████| 3135/3135 [24:58<00:00,  2.09it/s]


Época [5/10] | Train Loss: 0.0160 | Val Loss: 0.0206 | Val MAE: 0.0379 | Val CCC: 0.0045 | LR: 1.00e-04
Sem melhora por 4/5 épocas.


Época 6/10 [val]: 100%|██████████| 3135/3135 [25:01<00:00,  2.09it/s]

Época [6/10] | Train Loss: 0.0160 | Val Loss: 0.0208 | Val MAE: 0.0379 | Val CCC: -0.0038 | LR: 1.00e-04
Sem melhora por 5/5 épocas.
Early stopping ativado.

Treinamento concluído. Melhor CCC: 0.0101


In [ ]:
model = ResNetLSTM(
    frame_step=5,
    hidden_size=256,
    num_layers=2,
    use_dropout=True,
    dropout_p=0.3,
).to(device)

criterion = nn.MSELoss()
optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.5, verbose=True)

print(f"Parâmetros treináveis: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

train_model(
    model=model,
    train_loader=train_video_loader,
    val_loader=valid_video_loader,
    optimizer=optimizer,
    criterion=criterion,
    scheduler=scheduler,
    epochs=20,
    device=device,
    patience=5,
    checkpoint_path="/mnt/storage_C4/gaussian_football/models/checkpoints/resnet_lstm.pth",
)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/al.pedro.santos/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 89.5MB/s]
/mnt/storage_C4/gaussian_football/.venv/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Parâmetros treináveis: 14,389,953
TensorBoard: runs/arousal_20260606-145450


Época 1/50 [treino]:  53%|█████▎    | 5014/9405 [42:39<37:21,  1.96it/s]  


KeyboardInterrupt: 